## Evaluation — mini golden set (8 patients)

- **Exact match fields**: accuracy = correct / 8 across all patients
- **Jaccard fields**: IoU per patient, then mean across patients

In [7]:
import json
from pathlib import Path

golden_list      = json.loads(Path("../data/mini_golden_set.json").read_text())
postprocessed    = json.loads(Path("../data/results_postprocessed.json").read_text())

golden_by_id     = {g["patient_id"]: g for g in golden_list}
predicted_by_id  = {r["metadata"]["patient_id"]: r for r in postprocessed}

# Only evaluate patients present in both
patient_ids = sorted(golden_by_id.keys() & predicted_by_id.keys())
print(f"Evaluating {len(patient_ids)} patients: {patient_ids}")

Evaluating 9 patients: ['P004', 'P005', 'P010', 'P020', 'P029', 'P033', 'P034', 'P040', 'P047']


In [8]:
def flatten_golden(g: dict) -> dict:
    return {
        "completeness":          g["completeness"],
        "on_biologic":           g["on_biologic"],
        "biologic_name":         g["biologic_name"],
        "age":                   g["age"],
        "sex":                   g["sex"],
        "years_with_crohns":     g["years_with_crohns"],
        "biologic_status_detail": g["biologic_status_detail"],
        "pathway_endpoint":      g["pathway_endpoint"],
        "treatment_names":       {t["name"].lower() for t in g["treatment_journey"]},
        "barrier_categories":    set(g["barrier_categories"]) if g["barrier_categories"] else set(),
        "referral_pathway":      set(g["referral_pathway"]),
    }


def flatten_predicted(r: dict) -> dict:
    c1   = r["call1"]
    c2   = r.get("call2")
    c3   = r["call3"]
    return {
        "completeness":          c1["churn_detection"]["completeness"],
        "on_biologic":           c1["biologic_status"]["on_biologic"],
        "biologic_name":         c1["biologic_status"]["name"],
        "age":                   c1["sociodemographics"]["age"]["value"],
        "sex":                   c1["sociodemographics"]["sex"]["value"],
        "years_with_crohns":     c1["sociodemographics"]["years_with_crohns"]["value"],
        "biologic_status_detail": c2["biologic_status_detail"]["status"] if c2 else None,
        "pathway_endpoint":      c3["pathway_endpoint"]["status"],
        "treatment_names":       {t["name"].lower() for t in c1["treatment_journey"]},
        "barrier_categories":    set(r["category"] for r in c2["reasons"]) if c2 and c2.get("reasons") else set(),
        "referral_pathway":      {s["canonical_provider"] for s in c3["referral_pathway"]},
    }


flat_gold = {pid: flatten_golden(golden_by_id[pid])    for pid in patient_ids}
flat_pred = {pid: flatten_predicted(predicted_by_id[pid]) for pid in patient_ids}
print("Flattened OK")

Flattened OK


### Exact match accuracy

In [9]:
EXACT_FIELDS = [
    "completeness",
    "on_biologic",
    "biologic_name",
    "age",
    "sex",
    "years_with_crohns",
    "biologic_status_detail",
    "pathway_endpoint",
]

n = len(patient_ids)
print(f"{'Field':<25} {'Correct':>7}  {'Accuracy':>8}")
print("-" * 44)
for field in EXACT_FIELDS:
    correct = sum(1 for pid in patient_ids if flat_gold[pid][field] == flat_pred[pid][field])
    print(f"{field:<25} {correct:>4}/{n:<3}  {correct/n:>7.0%}")

Field                     Correct  Accuracy
--------------------------------------------
completeness                 9/9       100%
on_biologic                  9/9       100%
biologic_name                9/9       100%
age                          9/9       100%
sex                          9/9       100%
years_with_crohns            9/9       100%
biologic_status_detail       8/9        89%
pathway_endpoint             8/9        89%


### Per-field breakdown (exact match)

Shows gold vs predicted for each patient — useful for spotting systematic errors.

In [10]:
for field in EXACT_FIELDS:
    mismatches = [
        (pid, flat_gold[pid][field], flat_pred[pid][field])
        for pid in patient_ids
        if flat_gold[pid][field] != flat_pred[pid][field]
    ]
    if mismatches:
        print(f"\n{field} — {len(mismatches)} mismatch(es):")
        for pid, gold_val, pred_val in mismatches:
            print(f"  {pid}  gold={gold_val!r}  pred={pred_val!r}")
    else:
        print(f"\n{field} — ✅ all correct")


completeness — ✅ all correct

on_biologic — ✅ all correct

biologic_name — ✅ all correct

age — ✅ all correct

sex — ✅ all correct

years_with_crohns — ✅ all correct

biologic_status_detail — 1 mismatch(es):
  P047  gold='not_reached'  pred='not_mentioned'

pathway_endpoint — 1 mismatch(es):
  P047  gold='not_reached'  pred='discussed_no_decision'


### Jaccard similarity

In [11]:
def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0   # both empty → perfect agreement
    if not a or not b:
        return 0.0   # one empty → no overlap
    return len(a & b) / len(a | b)


JACCARD_FIELDS = ["treatment_names", "barrier_categories", "referral_pathway"]

print(f"{'Field':<22} ", end="")
for pid in patient_ids:
    print(f"{pid:>7}", end="")
print(f"  {'Mean':>7}")
print("-" * (22 + 7 * len(patient_ids) + 10))

for field in JACCARD_FIELDS:
    scores = [jaccard(flat_gold[pid][field], flat_pred[pid][field]) for pid in patient_ids]
    print(f"{field:<22} ", end="")
    for s in scores:
        print(f"{s:>7.2f}", end="")
    print(f"  {sum(scores)/len(scores):>7.2f}")

Field                     P004   P005   P010   P020   P029   P033   P034   P040   P047     Mean
-----------------------------------------------------------------------------------------------
treatment_names           1.00   1.00   1.00   1.00   0.78   0.83   0.83   1.00   1.00     0.94
barrier_categories        1.00   1.00   1.00   1.00   1.00   1.00   1.00   1.00   1.00     1.00
referral_pathway          1.00   0.50   1.00   0.50   1.00   0.50   1.00   0.50   1.00     0.78


### Jaccard breakdown

Shows which items were missed or added for each patient.

In [12]:
for field in JACCARD_FIELDS:
    print(f"\n{'='*60}\n{field}")
    for pid in patient_ids:
        gold = flat_gold[pid][field]
        pred = flat_pred[pid][field]
        score = jaccard(gold, pred)
        missed  = gold - pred   # in gold, not in pred
        extra   = pred - gold   # in pred, not in gold
        print(f"  {pid}  Jaccard={score:.2f}")
        if missed:
            print(f"    missed : {sorted(missed)}")
        if extra:
            print(f"    extra  : {sorted(extra)}")


treatment_names
  P004  Jaccard=1.00
  P005  Jaccard=1.00
  P010  Jaccard=1.00
  P020  Jaccard=1.00
  P029  Jaccard=0.78
    missed : ['combo therapy (stelara + adjunct)', 'fecal microbiota transplant trials']
  P033  Jaccard=0.83
    extra  : ['lamotrigine']
  P034  Jaccard=0.83
    missed : ['immunomodulator']
  P040  Jaccard=1.00
  P047  Jaccard=1.00

barrier_categories
  P004  Jaccard=1.00
  P005  Jaccard=1.00
  P010  Jaccard=1.00
  P020  Jaccard=1.00
  P029  Jaccard=1.00
  P033  Jaccard=1.00
  P034  Jaccard=1.00
  P040  Jaccard=1.00
  P047  Jaccard=1.00

referral_pathway
  P004  Jaccard=1.00
  P005  Jaccard=0.50
    extra  : ['GP']
  P010  Jaccard=1.00
  P020  Jaccard=0.50
    missed : ['Gastroenterologist']
  P029  Jaccard=1.00
  P033  Jaccard=0.50
    missed : ['Gastroenterologist']
  P034  Jaccard=1.00
  P040  Jaccard=0.50
    missed : ['Gastroenterologist']
  P047  Jaccard=1.00
